# AP 155 Lab Exercise
---
## Exercise 5: Matrices

_Instructions_: 
- **Report figure:** No report figure. Points in the report figure will be allocated to the report discussion 
- **Report discussion:**
  - What were the basis linear equations used to set up the matrix in each question of Problem 2?
  - Once you set up the matrices, what function did you use to solve the matrix equation (Could have different answers)? Check the docstring/documentation/other references and summarize the mechanism behind the function 
- **Code**: Complete the code and ensure it is functional, error-free, readable, and efficient (where needed). Include concise Markdown documentation highlighting the critical steps of your algorithm for each problem.

### Student Information
- _Full Name (Last Name, First Name)_: Buhangin, Ahrystan Deniz
- _Student No._: 2024-07766
- _Section_: THR-TX-1

### Grading Information (c/o Lab Instructor)
- [Rubrics description link](https://drive.google.com/file/d/1BMSlPot2Mc7XLu0eo4S8I8gLBsIadbCL/view?usp=sharing) (Note: percentages may still be tweaked)

| Criteria | Score | Subtotal |
| --- | --- | --- |
| Report figure | XX | 20 |
| Report discussion | XX | 20|
| Code readability | XX | 20 |
| Code efficiency | XX | 20 | 
| Code appropriateness  | XX | 20 | 
| **TOTAL** | XXX | 100 |

_Date and Time Scored (MM/DD/YYYY HH:MM AM/PM):_:_

---
## Section 1: Report

<!-- ### Report figure

<img src="figures/cat.jpg" alt="Alt text" width=45%> <img src="figures/cat.jpg" alt="Alt text" width=45%>


EDIT ME. See instruction above for what to include. Insert caption here, a short description what the figure illustrates. The syntax in markdown for putting a figure is `![Description](local_file_name.png)` or `<img src="/path/to_image.jpg" alt="Alt text" width=45%>`. Make sure the figure has complete elements.
-->

### Report discussion

Both items in #2 used a summation of forces to construct basis equations for a matrix. Specifically, the first question used component decomposition of tension forces along with the equilibrium condition to form two equations for two unknowns T1, and T2. The second question utilized torque, along with the stated constraints to form four equations for the four unkowns m1, m2, m3, and m4. 

I used numPy's linalg.solve for both items in problem 2. The linalg.solve function relies on the LAPACK _gesv routine which essentially uses LU decomposition to factor matrices into lower and upper triangular matrices. Solution is then completed via efficient forward and backward substitutions. As for problem 1, I stumbled upon SciPy's linalg.solve_banded function. This function intakes a compacted matrix which contains only the non-zero elements of a banded matrix and solves the system at a much more time and storage efficient manner.  

### Other comments
- EDIT ME
- A self reflection can be placed here.
- This portion is not graded.
- You may insert further questions here.


---
## Section 2: Code

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
import scipy as sc

### Problem 1: (Circuit Analysis)


Consider a long chain of resistors wired up like this:

<img src="reschain.png" alt="Alt text" width=95%>

All the resistors have the same resistance~$R$.  The power rail at the top is at voltage~$V_+=5$V.  The problem is to find the voltages $V_1...V_N$ at the internal points in the circuit.

1. **Using Ohm's law and the Kirchhoff current law**, which says that the total net current flow out of (or into) any junction in a circuit must be zero, show that the voltages $V_1\ldots V_N$ satisfy the equations

$$3V_1 - V_2 - V_3 = V_+ $$
$$-V_1 + 4V_2 - V_3 - V_4 = V_+ $$
      
$$-V_{i-2} - V_{i-1} + 4V_i - V_{i+1} - V_{i+2} = 0 $$

$$-V_{N-3} - V_{N-2} + 4V_{N-1} - V_N = V_- = 0 $$
$$-V_{N-2} - V_{N-1} + 3V_N = V_- = 0 $$

Note: The assignment of $V_-$ is to make the equations symmetric

2. Express these equations in vector form~$A\vec{v} = \vec{w}$ and find the values of the matrix~$A$ and the vector~$\vec{w}$.

3. Write a program to solve for the values of the~$V_i$ when there are $N=6$ internal junctions with unknown voltages.  (Hint: All the values of $V_i$ should lie between zero and $5$V.  If they don't, something is wrong.)

4. Now repeat your calculation for the case where there are $N=10\,000$ internal junctions.  This part is not possible using standard tools like the *solve* function.  You need to make use of the fact that the matrix~$\vec{A}$ is banded.  

In [2]:
N = 10000

A = np.array([[3, -1, -1, 0, 0, 0],
             [-1, 4, -1, -1, 0, 0],
             [-1, -1, 4, -1, -1, 0],
             [0, -1, -1, 4, -1, -1],
             [0, 0, -1, -1, 4, -1],
             [0, 0, 0, -1, -1, 3]])

b = np.array([5, 5, 0, 0, 0, 0])
V_small = np.linalg.solve(A, b)
print("V1 to V6:", [f"{v:.4f}V" for v in V_small])

y = np.zeros((5, N))

# main diagonal
y[2, :] = 4.0
y[2, 0] = 3.0
y[2, -1] = 3.0

# first upper and lower diagonals
y[1, 1:] = -1.0  
y[3, :-1] = -1.0  

# second upper and lower diagonals
y[0, 2:] = -1.0  
y[4, :-2] = -1.0  

w = np.zeros(N)
w[0] = 5
w[1] = 5


V_large = sc.linalg.solve_banded((2, 2), y, w)

print("First 3 voltages:", [f"{v:.4f}V" for v in V_large[:3]])
print("Last 3 voltages:", [f"{v:.4f}V" for v in V_large[-3:]])
    

V1 to V6: ['3.7255V', '3.4314V', '2.7451V', '2.2549V', '1.5686V', '1.2745V']
First 3 voltages: ['4.9989V', '4.9986V', '4.9980V']
Last 3 voltages: ['0.0020V', '0.0014V', '0.0011V']


#### Problem 1 code summary

Numbers 2 and 3 employ the basic np.linalg.solve function, which is functional for relatively small scale matrix solutions. Number 4, on the other hand, required a way to compact the banded 10k x 10k matrix into a 5 x 10k matrix containing only the non zero elements of the original matrix. This was done in a way where the two upper, one main, and two lower diagonals each took up one row of the matrix, after which SciPy's linalg,solve_banded function was employed.

---

### Problem 2 (Statics)


Solve the following matrix problems. Set-up the linear equations that describe the problem then solve them 

Part 1: Suppose you have a hanging mass M supported by two ropes (angled by $\alpha$ and $\beta$ with respect to the ceiling). Make a function that calculates the tension on each of the ropes. Put your function in a python file and call it here in the notebook. Check that the answer by computer matches your own computations.

<del>
Part 2: 
Masses $m_1, m_2, m_3$, and $m_4$ lie on a $2.00$ [m] beam of negligible mass. They are located $0.20, 0.70, 1.10$, and $1.40$ [m] away from the left end of the beam. Determine the masses $m_1, m_2, m_3$, and $m_4$, given the following constraints: 

- The total mass is $8.00$ [kg]
- If a pivot is placed halfway, the beam will balance if a $1.05$ [kg] mass is placed on the right-end of the beam
- If a pivot is placed $1.20$ [m] away from the right end, then the beam would balance if a $550$ [g] mass is placed on the left end of the beam.
- If the system is split midway, the total mass on the left is $1.00$ [kg] heavier than the total mass on the right
</del>

In [3]:
import statics as st

st.getTensions()

array([69.36717523, 69.36717523])

In [4]:

def r(x):
    return abs(1-x)

def r1(x):
    return abs(0.8 - x)

x1, x2, x3, x4, x5, x6 = 0.2, 0.7, 1.1, 1.4, 2, 0
g = 1
m5, m6 = 1.05, 0.550
A = np.array([[1, 1, 1, 1], 
          [g*r(x1), g*r(x2), -g*r(x3), -g*r(x4)],
          [g*r1(x1), g*r1(x2), -g*r1(x3), -g*r1(x4)],
          [1, 1, -1, -1]])
print(A)
b = np.array([8, m5*g*r(x5), -m6*g*r1(x6), 1])
print(b)

np.linalg.solve(A, b)

[[ 1.   1.   1.   1. ]
 [ 0.8  0.3 -0.1 -0.4]
 [ 0.6  0.1 -0.3 -0.6]
 [ 1.   1.  -1.  -1. ]]
[ 8.    1.05 -0.44  1.  ]


LinAlgError: Singular matrix

#### Problem 2 code summary

The first item was completed using component decomposition of the tension forces, and their equivalence to the gravitational force of the mass due to static equilibrium. The second item was completed using the definition of torque, and the constraints given subsequently. Both were solved using the np.linalg.solve function as they were quite small scale calculations.

---

## Section 3 (Notes)

### Some codes for solving matrix problems

* solving matrix equations ($\bf{A}\vec{x} = \vec{b}$) -> `np.linalg.solve`
* LU-decomposition ($\bf{A} = \bf{L}\bf{U}$) -> `scipy.linalg.lu`
* matrix inversion ($\bf{A}^{-1}$) -> `np.linalg.inv`
* QR-decomposition ($\bf{A} = \bf{Q}\bf{R}$) -> `scipy.linalg.qr`
* eigenvalue problem ($\bf{A}\vec{x} = \lambda \vec{x}$) -> `np.linalg.eig`

Solving matrix problems by themselves isn't difficult. There is an abundance of linear algebra libraries that are ready to use: from the battle-tested ones (the classic [BLAS](https://www.netlib.org/blas/) and [LAPACK](https://www.netlib.org/lapack/)), HPC tools for large problems ([ScaLAPACK](https://netlib.org/scalapack/)), to hardware-accelerated ones often with GPUs ([cuBLAS](https://docs.nvidia.com/cuda/cublas/index.html) and [cuSolver](https://docs.nvidia.com/cuda/cusolver/index.html) for NVIDIA). 

The hardest part is actually composing the matrix. How do you transform a physical problem into a linear algebra problem?

* Circuit: https://personal.math.vt.edu/embree/cmda3606/chapter2.pdf
* Embree [main notes](https://personal.math.vt.edu/embree/cmda3606notes.pdf) + [lab manual](https://personal.math.vt.edu/embree/labman.pdf)

#### Sample Usage (Testing different codes)

In [ ]:
A = np.array([[1,0,2],[-2,-1,3],[0,3,5]])
B = np.array([[-2],[1],[-3]])


In [ ]:
sigma_z = [[1,0],[0,-1]]
sigma_y = [[0,-1j],[1j,0]]
sigma_x = [[0,1],[1,0]]

In [ ]:
ans = np.linalg.solve(sigma_x, sigma_y)
ans = np.linalg.inv(A) @ B


In [ ]:
from scipy.linalg import *
p, l, u = lu(A)
p, l, u

### Importing functions from a python file

In [ ]:
import samplePackage as sP

In [ ]:
testMat = sP.createZerosArray(4,5)

In [ ]:
testMat[1] = [1,2,3,4,5]

In [ ]:
testMat

### np.roll

In [ ]:
## Useful tips. Not required to solve the problems below!

# Suppose you have a Numpy array
x = np.zeros((4,4))
x[1,2] = 5
for row in x:
    print(row)

#rint(x[0,1:3])


# I can move all values to the left/right using roll
x_roll = np.roll(x, 1) # change 1 to -1, what happens? 
#x_roll2 = np.roll(x, -1)
print(x_roll)

x_roll2 = np.roll(x, 2)
print(x_roll2)